# 06 多變項分析 — 練習

用松柏護理之家退伍軍人症 line list 練習 Modified Poisson regression（adjusted RR）
和邏輯斯迴歸（adjusted OR），並比較兩者差異。

In [ ]:
# Google Colab setup -- 若在本機執行可跳過此 cell
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || True
    os.chdir('/content/python4epi')
    !pip install -q -e .

In [ ]:
import pathlib

import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import warnings

# -- CJK font setup --
for _font_dir in map(pathlib.Path, ["/usr/share/fonts", "/usr/local/share/fonts"]):
    if _font_dir.exists():
        for _fp in sorted(_font_dir.rglob("*")):
            if _fp.suffix.lower() in {".ttf", ".ttc", ".otf"} and (
                "CJK" in _fp.name or "WenQuanYi" in _fp.name or "wqy" in _fp.name
            ):
                try:
                    fm.fontManager.addfont(str(_fp))
                except Exception:
                    pass

plt.rcParams["font.sans-serif"] = [
    "Noto Sans CJK TC", "Noto Sans CJK SC", "Noto Sans CJK JP",
    "Noto Sans TC", "Microsoft JhengHei",
    "WenQuanYi Zen Hei", "SimHei", "Arial Unicode MS",
    "Heiti TC", "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False
plt.style.use("ggplot")
plt.rcParams["figure.dpi"] = 150

df = pd.read_csv("data/synthetic/legionella_outbreak.csv")
df["infected"] = (df["clinical_severity"] != "not_ill").astype(int)
fs_map = {"bedridden": 0, "assisted": 1, "independent": 2}
df["functional_score"] = df["functional_status"].map(fs_map)

## 題目 1：死亡預測——Crude RR vs Crude OR

換一個結果變項——改為預測**死亡**（`outcome == 'dead'`）。

1. 建立 `dead` 欄位（0/1）
2. 計算死亡率（case fatality rate），判斷：死亡率高還是低？OR 和 RR 會差很多嗎？
3. 對以下變項**同時**計算 crude RR（Modified Poisson）和 crude OR（logistic）：
   `age`, `comorbidity_chf`, `comorbidity_copd`, `immunosuppressed`,
   `clinical_severity`（用數值：mild=1, moderate=2, severe=3）
4. 整理成表格，比較 RR 和 OR 的差異。死亡率較低時，差距是否變小了？

In [ ]:
# TODO: 建立 dead 欄位
# TODO: 計算死亡率
# TODO: 迴圈同時計算 crude RR (Modified Poisson) 和 crude OR (logistic)
# TODO: 整理成 RR vs OR 對照表格

## 題目 2：多變項 Adjusted RR + Adjusted OR

建立一個預測死亡的多變項模型：

```
dead ~ age + comorbidity_chf + comorbidity_copd + immunosuppressed + severity_score
```

1. 用 **Modified Poisson**（`smf.glm(..., family=Poisson()).fit(cov_type='HC0')`）算 adjusted RR
2. 用 **Logistic Regression**（`smf.logit()`）算 adjusted OR
3. 並排比較 adjusted RR 和 adjusted OR，哪些變項差距最大？
4. 與題目 1 的 crude 結果比較，crude → adjusted 變化最大的是哪個變項？（→ 受干擾最多的因子）

In [ ]:
# TODO: Modified Poisson 多變項模型 → adjusted RR
# TODO: Logistic 多變項模型 → adjusted OR
# TODO: 並排比較表格
# TODO: 比較 crude vs adjusted

## 題目 3（挑戰題）：模型比較 + Forest Plot

1. 建立兩個 Modified Poisson 模型：
   - 模型 A：`dead ~ age + immunosuppressed + severity_score`
   - 模型 B：`dead ~ age + comorbidity_chf + comorbidity_copd + immunosuppressed + severity_score`
2. 比較 AIC，哪個較好？
3. 用較好的模型畫 **Adjusted RR** 森林圖（參考課堂筆記的 forest plot 程式碼）
4. 解讀：哪些因子是死亡的獨立預測因子（RR > 1 且 CI 不包含 1）？

In [ ]:
# TODO: 建立模型 A 和 B（Modified Poisson + HC0）
# TODO: 比較 AIC
# TODO: Forest plot（adjusted RR）
# TODO: 解讀

## 題目 4：結核病接觸者風險因子（結核 TB 情境）

某社區針對結核病患者的接觸者進行疫調與胸部 X 光篩檢，共收案 600 人，記錄是否確診為活動性肺結核（`active_tb`）。

1. 計算樣本中活動性結核的盛行率。
2. 計算「家戶密切接觸者」（`close_contact`）的 crude OR 及 95% 信賴區間。
3. 建立多變項邏輯斯迴歸模型：

   ```
   active_tb ~ age + diabetes + close_contact + underweight + smoking
   ```

   計算每個變項的 adjusted OR 及 95% CI。
4. 比較 `close_contact` 的 crude OR 與 adjusted OR，是否受到其他干擾因子影響？
5. 解讀：校正後有哪些危險因子的 95% CI 不含 1（達統計顯著）？adjusted OR 最大的是哪一個？

In [ ]:
# --- 資料：結核病接觸者疫調（合成資料）---
rng = np.random.default_rng(406)
n = 600

age = rng.normal(45, 15, n).clip(5, 90)
diabetes = (rng.uniform(0, 1, n) < (0.05 + age / 300)).astype(int)
close_contact = rng.binomial(1, 0.4, n)          # 是否為同住家戶密切接觸者
underweight = rng.binomial(1, 0.15, n)           # BMI < 18.5（營養不良）
smoking = rng.binomial(1, 0.25, n)

logit_p = (
    -4.3
    + 0.03 * age
    + 0.8 * diabetes
    + 1.3 * close_contact
    + 0.9 * underweight
    + 0.5 * smoking
)
p_tb = 1 / (1 + np.exp(-logit_p))
active_tb = rng.binomial(1, p_tb)

tb = pd.DataFrame({
    "age": age.round(1),
    "diabetes": diabetes,
    "close_contact": close_contact,
    "underweight": underweight,
    "smoking": smoking,
    "active_tb": active_tb,
})

# TODO: 計算 active_tb 的盛行率
# TODO: 用 smf.logit("active_tb ~ close_contact", data=tb) 計算 close_contact 的 crude OR 及 95% CI
# TODO: 建立多變項模型 active_tb ~ age + diabetes + close_contact + underweight + smoking，計算 adjusted OR
# TODO: 比較 close_contact 的 crude OR 與 adjusted OR
# TODO: 整理哪些變項的 95% CI 不含 1，並指出 adjusted OR 最大的危險因子

## 題目 5：COVID-19 重症預測（COVID-19 情境）

某醫院於社區篩檢站建立 COVID-19 確診病例名冊，共 600 筆，記錄是否發生重症（`severe`，需住院或加護病房）。

1. 計算重症比例。
2. 計算「未接種疫苗」（`unvaccinated`）的 crude OR 及 95% CI。
3. 建立多變項邏輯斯迴歸模型：

   ```
   severe ~ age + obesity + unvaccinated + chronic_lung
   ```

   計算每個變項的 adjusted OR 及 95% CI。
4. 比較 `unvaccinated` 的 crude OR 與 adjusted OR。
5. 解讀：`unvaccinated` 的 adjusted OR 是否顯著大於 1（95% CI 不含 1）？這對「接種疫苗能否降低重症風險」的結論有何意義？

In [ ]:
# --- 資料：COVID-19 社區篩檢確診病例（合成資料）---
rng = np.random.default_rng(507)
n = 600

age = rng.normal(50, 18, n).clip(18, 95)
obesity = rng.binomial(1, 0.30, n)
unvaccinated = rng.binomial(1, 0.35, n)
chronic_lung = rng.binomial(1, 0.12, n)

logit_p = (
    -5.3
    + 0.05 * age
    + 0.8 * obesity
    + 1.1 * unvaccinated
    + 0.9 * chronic_lung
)
p_severe = 1 / (1 + np.exp(-logit_p))
severe = rng.binomial(1, p_severe)

covid = pd.DataFrame({
    "age": age.round(1),
    "obesity": obesity,
    "unvaccinated": unvaccinated,
    "chronic_lung": chronic_lung,
    "severe": severe,
})

# TODO: 計算重症比例
# TODO: 計算 unvaccinated 的 crude OR 及 95% CI
# TODO: 建立多變項模型 severe ~ age + obesity + unvaccinated + chronic_lung，計算 adjusted OR
# TODO: 比較 crude OR 與 adjusted OR
# TODO: 解讀 unvaccinated 的 adjusted OR 是否顯著大於 1

## 題目 6：登革熱重症（DHF）危險因子（登革熱情境）

某縣市登革熱疫情期間，共收案 550 名確診病例，記錄是否發生重症登革熱（`severe_dengue`，即登革熱出血熱 DHF／登革休克症候群 DSS）。

1. 計算重症登革熱的比例。
2. 計算「二次感染」（`secondary_infection`，曾感染過不同血清型登革病毒）的 crude OR 及 95% CI。
3. 建立多變項邏輯斯迴歸模型：

   ```
   severe_dengue ~ secondary_infection + age + diabetes + hypertension
   ```

   計算每個變項的 adjusted OR 及 95% CI。
4. 比較 `secondary_infection` 的 crude OR 與 adjusted OR，是否受到年齡或共病干擾？
5. 解讀：根據 adjusted OR，二次感染者發生重症登革熱的勝算是初次感染者的幾倍？這是否與抗體依賴增強作用（antibody-dependent enhancement, ADE）的病理機轉一致？

In [ ]:
# --- 資料：登革熱確診病例（合成資料）---
rng = np.random.default_rng(608)
n = 550

secondary_infection = rng.binomial(1, 0.30, n)   # 是否為二次感染（不同血清型）
age = rng.normal(35, 20, n).clip(1, 85)
diabetes = rng.binomial(1, 0.15, n)
hypertension = rng.binomial(1, 0.20, n)

logit_p = (
    -3.8
    + 1.5 * secondary_infection
    + 0.03 * age
    + 0.7 * diabetes
    + 0.5 * hypertension
)
p_severe = 1 / (1 + np.exp(-logit_p))
severe_dengue = rng.binomial(1, p_severe)

dengue = pd.DataFrame({
    "secondary_infection": secondary_infection,
    "age": age.round(1),
    "diabetes": diabetes,
    "hypertension": hypertension,
    "severe_dengue": severe_dengue,
})

# TODO: 計算重症登革熱比例
# TODO: 計算 secondary_infection 的 crude OR 及 95% CI
# TODO: 建立多變項模型 severe_dengue ~ secondary_infection + age + diabetes + hypertension，計算 adjusted OR
# TODO: 比較 crude OR 與 adjusted OR，判斷是否受干擾
# TODO: 解讀 adjusted OR(secondary_infection) 的意義

## 題目 7：流感住院危險因子（流感情境）

某年流感季，社區診所建立確診病例名冊，共 700 筆，記錄確診後是否住院（`hospitalized`）。

1. 計算住院比例。
2. 計算「已接種流感疫苗」（`vaccinated`）的 crude OR 及 95% CI（提示：慢性病患者常被優先建議接種疫苗，crude OR 可能被干擾因子扭曲）。
3. 建立多變項邏輯斯迴歸模型：

   ```
   hospitalized ~ age + chronic_disease + vaccinated + late_treatment
   ```

   計算每個變項的 adjusted OR 及 95% CI。
4. 比較 `vaccinated` 的 crude OR 與 adjusted OR，方向是否在校正後改變（例如由「風險升高」變成「保護效果」）？
5. 解讀：這種因「高風險族群更常接受介入措施」而扭曲關聯方向的現象稱為什麼？為什麼疫苗接種與慢性病史之間的關聯會造成這種干擾模式？

In [ ]:
# --- 資料：流感確診病例（合成資料，含「適應症干擾」confounding by indication）---
rng = np.random.default_rng(709)
n = 700

age = rng.normal(40, 20, n).clip(0, 95)
chronic_disease = rng.binomial(1, 0.22, n)
# 慢性病患者常被優先建議接種疫苗 -> vaccinated 與 chronic_disease 強烈正相關（適應症干擾）
p_vaccinated = 0.20 + 0.65 * chronic_disease
vaccinated = rng.binomial(1, p_vaccinated)
late_treatment = rng.binomial(1, 0.40, n)   # 症狀 48 小時後才投藥

logit_p = (
    -4.1
    + 0.03 * age
    + 1.8 * chronic_disease
    - 0.65 * vaccinated
    + 0.9 * late_treatment
)
p_hosp = 1 / (1 + np.exp(-logit_p))
hospitalized = rng.binomial(1, p_hosp)

flu = pd.DataFrame({
    "age": age.round(1),
    "chronic_disease": chronic_disease,
    "vaccinated": vaccinated,
    "late_treatment": late_treatment,
    "hospitalized": hospitalized,
})

# TODO: 計算住院比例
# TODO: 計算 vaccinated 的 crude OR 及 95% CI
# TODO: 建立多變項模型 hospitalized ~ age + chronic_disease + vaccinated + late_treatment，計算 adjusted OR
# TODO: 比較 crude OR 與 adjusted OR，觀察方向是否改變
# TODO: 解讀此干擾現象（適應症干擾 confounding by indication）

## 題目 8（挑戰題）：麻疹重症併發症的交互作用（麻疹情境）

某麻疹群突發中，共收案 650 名確診病例（多為兒童），記錄是否發生重症併發症（`severe_complication`，如肺炎或腦炎）。已知營養不良與維生素 A 缺乏在病理機轉上可能有加乘作用。

1. 依 `malnutrition`、`vitamin_a_deficiency` 分成四組（皆無／僅營養不良／僅維生素A缺乏／兩者皆有），比較各組重症併發症的比例，觀察是否有加乘效應的跡象。
2. 建立不含交互作用項的主效應模型：

   ```
   severe_complication ~ malnutrition + vitamin_a_deficiency + age
   ```

   計算 adjusted OR。
3. 建立含交互作用項的模型：

   ```
   severe_complication ~ malnutrition * vitamin_a_deficiency + age
   ```

   （statsmodels 會自動展開為 `malnutrition + vitamin_a_deficiency + malnutrition:vitamin_a_deficiency`），計算交互作用項的 OR 及 95% CI。
4. 用 AIC 比較兩個模型，交互作用項是否改善模型適配度？
5. 解讀：交互作用項的 OR 若大於 1，在邏輯斯迴歸中代表什麼意義（相乘尺度上的加乘作用）？這對公衛政策（例如優先為營養不良兒童補充維生素 A）有何啟示？

In [ ]:
# --- 資料：麻疹群突發病例（合成資料，含 malnutrition x vitamin_a_deficiency 交互作用）---
rng = np.random.default_rng(810)
n = 650

age = rng.uniform(0.5, 15, n)                 # 兒童為主
malnutrition = rng.binomial(1, 0.25, n)
vitamin_a_deficiency = rng.binomial(1, 0.20, n)

logit_p = (
    -2.8
    + 0.6 * malnutrition
    + 0.6 * vitamin_a_deficiency
    + 1.6 * malnutrition * vitamin_a_deficiency   # 交互作用：兩者同時存在時風險加乘
    - 0.05 * age
)
p_severe = 1 / (1 + np.exp(-logit_p))
severe_complication = rng.binomial(1, p_severe)

measles = pd.DataFrame({
    "age": age.round(2),
    "malnutrition": malnutrition,
    "vitamin_a_deficiency": vitamin_a_deficiency,
    "severe_complication": severe_complication,
})

# TODO: 分成四組（皆無/僅營養不良/僅維生素A缺乏/兩者皆有），比較各組重症比例
# TODO: 建立主效應模型 severe_complication ~ malnutrition + vitamin_a_deficiency + age
# TODO: 建立交互作用模型 severe_complication ~ malnutrition * vitamin_a_deficiency + age
# TODO: 比較兩模型 AIC
# TODO: 解讀交互作用項的 OR 及其公衛意義